# KG1 V227 Targeted Equation Micro-Sweep Colab

Purpose: move past the V226 `191/315` ceiling by evaluating V194/V217/V223/V226 checkpoints and, when enabled, creating a tiny targeted V227 continuation from the best observed V226 checkpoint-1.

The objective is explicit: find a weak candidate with at least `193/315` total, `60/155` equation_transform, `133/160` bit_manipulation, and no more than `3` truncations. Full eval, packaging, and Kaggle submit remain blocked in this notebook.

Colab: https://colab.research.google.com/github/FELIPEACASTRO/KG1-NVIDIA/blob/v227-targeted-equation-micro-sweep/notebooks/KG1_V227_TARGETED_EQUATION_MICRO_SWEEP_COLAB.ipynb

GitHub: https://github.com/FELIPEACASTRO/KG1-NVIDIA/blob/v227-targeted-equation-micro-sweep/notebooks/KG1_V227_TARGETED_EQUATION_MICRO_SWEEP_COLAB.ipynb


In [1]:
# CELL: mount Google Drive.
print('=== V227 DRIVE MOUNT START ===', flush=True)
from google.colab import drive
drive.mount('/content/drive')
print('=== V227 DRIVE MOUNT END ===', flush=True)


=== V227 DRIVE MOUNT START ===
Mounted at /content/drive
=== V227 DRIVE MOUNT END ===


In [2]:
# CELL: global configuration, gates, and hard submit lock.
print('=== V227 CONFIG START ===', flush=True)
import datetime
import hashlib
import importlib
import json
import os
import pathlib
import shutil
import subprocess
import sys
import time

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('BITSANDBYTES_NOWELCOME', '1')
os.environ.setdefault('KG1_ALLOW_VLLM_DEEP_GEMM', '0')
os.environ.setdefault('VLLM_USE_DEEP_GEMM', '0')
os.environ.setdefault('VLLM_MOE_USE_DEEP_GEMM', '0')
os.environ.setdefault('VLLM_USE_DEEP_GEMM_E8M0', '0')
os.environ.setdefault('VLLM_USE_DEEP_GEMM_TMA_ALIGNED_SCALES', '0')
os.environ.setdefault('VLLM_DEEP_GEMM_WARMUP', 'skip')
os.environ.setdefault('VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS', '0')
os.environ.setdefault('TORCH_CUDA_ARCH_LIST', os.environ.get('KG1_TORCH_CUDA_ARCH_LIST', '9.0'))
os.environ.setdefault('MAX_JOBS', os.environ.get('KG1_BUILD_MAX_JOBS', '4'))

try:
    from google.colab import userdata
    for secret_name in ['HF_TOKEN', 'HF_KEY']:
        if not os.environ.get('HF_TOKEN'):
            secret_value = userdata.get(secret_name)
            if secret_value:
                os.environ['HF_TOKEN'] = secret_value
                os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_value
                print('loaded Hugging Face token from Colab secret:', secret_name, flush=True)
except Exception as exc:
    print('Colab secret probe skipped:', type(exc).__name__, flush=True)

VERSION = 'V227_TARGETED_EQUATION_MICRO_SWEEP_20260509'
REPO_URL = os.environ.get('KG1_REPO_URL', 'https://github.com/FELIPEACASTRO/KG1-NVIDIA.git')
REPO_BRANCH = os.environ.get('KG1_REPO_BRANCH', 'v227-targeted-equation-micro-sweep')
ROOT = pathlib.Path('/content/kg1')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V227')
OUT_ROOT = DRIVE_ROOT / 'output_v227_targeted_equation_micro_sweep'
TRAIN_OUT = OUT_ROOT / 'train_v227_from_v226ckpt1_eq_nudge_lr5e10_s2'
EVAL_OUT = OUT_ROOT / 'eval_v227_targeted_equation_micro_sweep'
ANALYSIS_OUT = OUT_ROOT / 'analysis_v227_targeted_equation_micro_sweep'

MODEL_NAME = os.environ.get('KG1_V227_MODEL_NAME', 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16')
V227_VLLM_PIP_SPEC = os.environ.get('KG1_V227_VLLM_PIP_SPEC', 'vllm==0.20.1')
V227_CAUSAL_CONV1D_PIP_SPEC = os.environ.get('KG1_V227_CAUSAL_CONV1D_PIP_SPEC', 'causal-conv1d==1.6.1')
V227_MAMBA_SSM_PIP_SPEC = os.environ.get('KG1_V227_MAMBA_SSM_PIP_SPEC', 'mamba-ssm==2.3.1')
V227_MAX_MODEL_LEN = int(os.environ.get('KG1_V227_MAX_MODEL_LEN', '8192'))
V227_MAX_NUM_SEQS = int(os.environ.get('KG1_V227_MAX_NUM_SEQS', '64'))
V227_MAX_TOKENS = int(os.environ.get('KG1_V227_MAX_TOKENS', '7680'))
V227_WARMUP_ROWS = int(os.environ.get('KG1_V227_WARMUP_ROWS', '0'))
V227_PROMPT_SUFFIX = os.environ.get('KG1_V227_PROMPT_SUFFIX', '\nReturn exactly one line in this format: `\\boxed{answer}`.')

V221_EVAL_OUT = pathlib.Path(os.environ.get('KG1_V227_V221_EVAL_OUT', '/content/drive/MyDrive/KG1_NVIDIA_V221/output_v221_candidate_registry_weak_ab/eval_v221_candidate_registry_weak_ab'))
V221_WEAK_CSV = pathlib.Path(os.environ.get('KG1_V227_V221_WEAK_CSV', str(V221_EVAL_OUT / 'v221_weak_315.csv')))
V221_BATCH_SUMMARY_JSON = pathlib.Path(os.environ.get('KG1_V227_V221_BATCH_SUMMARY_JSON', str(V221_EVAL_OUT / 'batch_candidate_summary.json')))
V225_FINAL_MANIFEST = pathlib.Path(os.environ.get('KG1_V227_V225_FINAL_MANIFEST', '/content/drive/MyDrive/KG1_NVIDIA_V225/output_v225_equation_decode_sweep/v225_equation_decode_sweep_final_manifest.json'))
V223_TRAIN_OUT = pathlib.Path(os.environ.get('KG1_V227_V223_TRAIN_OUT', '/content/drive/MyDrive/KG1_NVIDIA_V223/output_v223_equation_rescue/train_v223_eqrescue_from_v217_lr1e8_s12'))
V226_TRAIN_OUT = pathlib.Path(os.environ.get('KG1_V227_V226_TRAIN_OUT', '/content/drive/MyDrive/KG1_NVIDIA_V226/output_v226_equation_checkpoint_sweep/train_v226_v194_micro_lr2e9_s6'))
V226_BEST_CHECKPOINT = pathlib.Path(os.environ.get('KG1_V227_V226_BEST_CHECKPOINT', str(V226_TRAIN_OUT / 'checkpoint-1')))

V194_ADAPTER = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V202D/init_adapter_v194_rank19_build/adapter')
V217_ADAPTER = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V217/output_v217_short_answer_rescue/train_v217_shortans_lr1e8_s16/final_adapter')
INIT_ADAPTER_DIR = pathlib.Path(os.environ.get('KG1_V227_INIT_ADAPTER', str(V226_BEST_CHECKPOINT)))
EXPECTED_TRAIN_SHA256 = 'a56938b1ae9eb471b779ebfc415ee88c05322941732128752680317495157984'
EXPECTED_VAL_SHA256 = '65c4cb88b8ff2fc96940ccea33b8ca493769790c7ae80d27f2b69ac818fc6451'
MIN_TRAIN_EXAMPLES = 10206
MIN_VAL_EXAMPLES = 681
TOKENIZE_ONLY_DRY_RUN = True
MAX_PROMPT_TRUNCATION_RATE = 0.0
REQUIRE_OFFSET_MASK = True

EXPECTED_V194_ADAPTER_BYTES = 4259069440
EXPECTED_V194_ADAPTER_TENSOR_COUNT = 12011
MIN_V217_ADAPTER_BYTES = 4250000000
MIN_V217_ADAPTER_TENSOR_COUNT = 12000
EXPECTED_V194_TARGET_MODULES = ['k_proj', 'up_proj', 'down_proj', 'out_proj', 'v_proj', 'q_proj', 'lm_head', 'o_proj', 'in_proj']
EXPECTED_V194_TARGET_PARAMETERS = ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']

RUN_TRAIN = os.environ.get('KG1_V227_RUN_TRAIN', '0').strip().lower() in {'1', 'true', 'yes', 'on'}
RUN_EVAL = os.environ.get('KG1_V227_RUN_EVAL', '1').strip().lower() not in {'0', 'false', 'no', 'off'}
RUN_ANALYSIS = os.environ.get('KG1_V227_RUN_ANALYSIS', '1').strip().lower() not in {'0', 'false', 'no', 'off'}
RUN_FULL_IF_GATE = False
FORCE_RETRAIN = os.environ.get('KG1_V227_FORCE_RETRAIN', '0').strip().lower() in {'1', 'true', 'yes', 'on'}
FORCE_REEVAL = os.environ.get('KG1_V227_FORCE_REEVAL', '0').strip().lower() in {'1', 'true', 'yes', 'on'}

V227_LR = os.environ.get('KG1_V227_LR', '5e-10')
V227_FINAL_LR = os.environ.get('KG1_V227_FINAL_LR', '1e-10')
V227_MAX_STEPS = os.environ.get('KG1_V227_MAX_STEPS', '2')
V227_TRAIN_SEED = os.environ.get('KG1_V227_TRAIN_SEED', '91')
V227_TRAINABLE_MODULES = os.environ.get('KG1_V227_TRAINABLE_MODULES', 'lm_head,o_proj,q_proj,k_proj')
V227_MAX_LENGTH = int(os.environ.get('KG1_V227_MAX_LENGTH', '4096'))
V227_BATCH_SIZE = int(os.environ.get('KG1_V227_BATCH_SIZE', '4'))
V227_MICRO_BATCH_SIZE = int(os.environ.get('KG1_V227_MICRO_BATCH_SIZE', '1'))
V227_MAX_CANDIDATES = int(os.environ.get('KG1_V227_MAX_CANDIDATES', '32'))

WEAK_MIN_FOR_FULL = 193
WEAK_EQ_MIN_FOR_FULL = 60
WEAK_BIT_MIN_FOR_FULL = 133
WEAK_MAX_TRUNC_FOR_FULL = 3
FULL_MIN_CANDIDATE = 831
FULL_MAX_TRUNC = 4
ALLOW_KAGGLE_SUBMIT = False

for path in [DRIVE_ROOT, OUT_ROOT, TRAIN_OUT, EVAL_OUT, ANALYSIS_OUT]:
    path.mkdir(parents=True, exist_ok=True)

print('VERSION =', VERSION, flush=True)
print('REPO_URL =', REPO_URL, flush=True)
print('REPO_BRANCH =', REPO_BRANCH, flush=True)
print('ROOT =', ROOT, flush=True)
print('OUT_ROOT =', OUT_ROOT, flush=True)
print('TRAIN_OUT =', TRAIN_OUT, flush=True)
print('EVAL_OUT =', EVAL_OUT, flush=True)
print('ANALYSIS_OUT =', ANALYSIS_OUT, flush=True)
print('MODEL_NAME =', MODEL_NAME, flush=True)
print('V227_MAX_TOKENS =', V227_MAX_TOKENS, flush=True)
print('V227_PROMPT_SUFFIX =', repr(V227_PROMPT_SUFFIX), flush=True)
print('V221_WEAK_CSV =', V221_WEAK_CSV, flush=True)
print('V221_BATCH_SUMMARY_JSON =', V221_BATCH_SUMMARY_JSON, flush=True)
print('V225_FINAL_MANIFEST =', V225_FINAL_MANIFEST, flush=True)
print('V223_TRAIN_OUT =', V223_TRAIN_OUT, flush=True)
print('V226_TRAIN_OUT =', V226_TRAIN_OUT, flush=True)
print('V226_BEST_CHECKPOINT =', V226_BEST_CHECKPOINT, flush=True)
print('V194_ADAPTER =', V194_ADAPTER, flush=True)
print('V217_ADAPTER =', V217_ADAPTER, flush=True)
print('INIT_ADAPTER_DIR =', INIT_ADAPTER_DIR, flush=True)
print('EXPECTED_TRAIN_SHA256 =', EXPECTED_TRAIN_SHA256, flush=True)
print('EXPECTED_VAL_SHA256 =', EXPECTED_VAL_SHA256, flush=True)
print('TOKENIZE_ONLY_DRY_RUN =', TOKENIZE_ONLY_DRY_RUN, flush=True)
print('MAX_PROMPT_TRUNCATION_RATE =', MAX_PROMPT_TRUNCATION_RATE, flush=True)
print('REQUIRE_OFFSET_MASK =', REQUIRE_OFFSET_MASK, flush=True)
print('RUN_TRAIN =', RUN_TRAIN, flush=True)
print('RUN_EVAL =', RUN_EVAL, flush=True)
print('RUN_ANALYSIS =', RUN_ANALYSIS, flush=True)
print('RUN_FULL_IF_GATE =', RUN_FULL_IF_GATE, flush=True)
print('FORCE_RETRAIN =', FORCE_RETRAIN, flush=True)
print('FORCE_REEVAL =', FORCE_REEVAL, flush=True)
print('V227_LR =', V227_LR, flush=True)
print('V227_FINAL_LR =', V227_FINAL_LR, flush=True)
print('V227_MAX_STEPS =', V227_MAX_STEPS, flush=True)
print('V227_TRAIN_SEED =', V227_TRAIN_SEED, flush=True)
print('V227_TRAINABLE_MODULES =', V227_TRAINABLE_MODULES, flush=True)
print('V227_MAX_CANDIDATES =', V227_MAX_CANDIDATES, flush=True)
print('WEAK_MIN_FOR_FULL =', WEAK_MIN_FOR_FULL, flush=True)
print('WEAK_EQ_MIN_FOR_FULL =', WEAK_EQ_MIN_FOR_FULL, flush=True)
print('WEAK_BIT_MIN_FOR_FULL =', WEAK_BIT_MIN_FOR_FULL, flush=True)
print('WEAK_MAX_TRUNC_FOR_FULL =', WEAK_MAX_TRUNC_FOR_FULL, flush=True)
print('FULL_MIN_CANDIDATE =', FULL_MIN_CANDIDATE, flush=True)
print('FULL_MAX_TRUNC =', FULL_MAX_TRUNC, flush=True)
print('ALLOW_KAGGLE_SUBMIT =', ALLOW_KAGGLE_SUBMIT, flush=True)
if ALLOW_KAGGLE_SUBMIT:
    raise RuntimeError('Kaggle submission is disabled in V227.')
print('=== V227 CONFIG END ===', flush=True)


=== V227 CONFIG START ===
loaded Hugging Face token from Colab secret: HF_TOKEN
VERSION = V227_TARGETED_EQUATION_MICRO_SWEEP_20260509
REPO_URL = https://github.com/FELIPEACASTRO/KG1-NVIDIA.git
REPO_BRANCH = v227-targeted-equation-micro-sweep
ROOT = /content/kg1
OUT_ROOT = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep
TRAIN_OUT = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/train_v227_from_v226ckpt1_eq_nudge_lr5e10_s2
EVAL_OUT = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/eval_v227_targeted_equation_micro_sweep
ANALYSIS_OUT = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/analysis_v227_targeted_equation_micro_sweep
MODEL_NAME = nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
V227_MAX_TOKENS = 7680
V227_PROMPT_SUFFIX = '\nReturn exactly one line in this format: `\\boxed{answer}`.'
V221_WEAK_CSV = /content/drive/MyDrive/KG1_NVIDIA_V221/output_v221_candid

In [3]:
# CELL: helper functions with command logging, heartbeat, hashes, and dependency installers.
print('=== V227 HELPERS START ===', flush=True)

def sha256_file(path):
    path = pathlib.Path(path)
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(pathlib.Path(path).read_text(encoding='utf-8'))

def write_json(path, payload):
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding='utf-8')

def resource_snapshot_line():
    parts = []
    try:
        import psutil
        mem = psutil.virtual_memory()
        parts.append(f'ram_total={mem.total/1024**3:.1f}GiB')
        parts.append(f'ram_available={mem.available/1024**3:.1f}GiB')
    except Exception as exc:
        parts.append(f'ram_probe_error={type(exc).__name__}')
    try:
        usage = shutil.disk_usage('/content')
        parts.append(f'disk_content_free={usage.free/1024**3:.1f}GiB')
        parts.append(f'disk_content_total={usage.total/1024**3:.1f}GiB')
    except Exception as exc:
        parts.append(f'disk_probe_error={type(exc).__name__}')
    try:
        gpu_line = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.used,memory.total,utilization.gpu', '--format=csv,noheader,nounits'],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            check=False,
        ).stdout.strip().splitlines()
        if gpu_line:
            parts.append('gpu=[' + gpu_line[0] + ']')
    except Exception as exc:
        parts.append(f'gpu_probe_error={type(exc).__name__}')
    return ' '.join(parts)

def run_cmd(cmd, cwd=None, log_path=None, check=True, heartbeat_s=0, suppress_after_lines=260):
    started = time.time()
    printable = ' '.join(str(x) for x in cmd)
    print('--- COMMAND START ---', flush=True)
    print('cwd =', cwd or os.getcwd(), flush=True)
    print('+', printable, flush=True)
    log_handle = None
    if log_path is not None:
        log_path = pathlib.Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_handle = log_path.open('w', encoding='utf-8', errors='replace')
        print('log_path =', log_path, flush=True)
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd is not None else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    last_heartbeat = time.time()
    assert proc.stdout is not None
    for line in proc.stdout:
        lines.append(line.rstrip('\n'))
        if log_handle:
            log_handle.write(line)
            log_handle.flush()
        if len(lines) <= suppress_after_lines:
            print(line, end='', flush=True)
        now = time.time()
        if heartbeat_s and now - last_heartbeat >= heartbeat_s:
            print('[V227 heartbeat] elapsed_s={:.1f} {}'.format(now - started, resource_snapshot_line()), flush=True)
            last_heartbeat = now
    returncode = proc.wait()
    if log_handle:
        log_handle.close()
    if len(lines) > suppress_after_lines:
        print('command_output_suppressed_lines =', len(lines) - suppress_after_lines, flush=True)
    elapsed = time.time() - started
    print('returncode =', returncode, flush=True)
    print('elapsed_s =', round(elapsed, 1), flush=True)
    if returncode != 0:
        print('command_tail_on_failure =', '\n'.join(lines[-60:]), flush=True)
    print('--- COMMAND END ---', flush=True)
    if check and returncode != 0:
        raise RuntimeError(f'command failed rc={returncode}: {printable}')
    return returncode

def verify_import(module_name, log_name):
    cmd = [
        sys.executable,
        '-c',
        "import importlib; m=importlib.import_module(%r); print(%r + ' subprocess_version=' + str(getattr(m, '__version__', 'unknown')))" % (module_name, module_name),
    ]
    return run_cmd(cmd, cwd='/content', log_path=OUT_ROOT / log_name, check=False)

def ensure_train_dependencies():
    print('=== V227 TRAIN DEPENDENCY CHECK START ===', flush=True)
    if verify_import('causal_conv1d', 'verify_import_causal_conv1d.log') != 0:
        print('installing causal_conv1d =', V227_CAUSAL_CONV1D_PIP_SPEC, flush=True)
        run_cmd([sys.executable, '-m', 'pip', 'install', '--progress-bar', 'off', '--no-build-isolation', V227_CAUSAL_CONV1D_PIP_SPEC], cwd='/content', log_path=OUT_ROOT / 'pip_install_causal_conv1d.log', check=True, heartbeat_s=60)
    if verify_import('mamba_ssm', 'verify_import_mamba_ssm.log') != 0:
        print('installing mamba_ssm =', V227_MAMBA_SSM_PIP_SPEC, flush=True)
        run_cmd([sys.executable, '-m', 'pip', 'install', '--progress-bar', 'off', '--no-build-isolation', V227_MAMBA_SSM_PIP_SPEC], cwd='/content', log_path=OUT_ROOT / 'pip_install_mamba_ssm.log', check=True, heartbeat_s=60)
    print('=== V227 TRAIN DEPENDENCY CHECK END ===', flush=True)

def ensure_vllm_for_eval():
    print('=== V227 VLLM EVAL DEPENDENCY CHECK START ===', flush=True)
    if verify_import('vllm', 'verify_import_vllm.log') != 0:
        print('vLLM subprocess import failed; installing pinned V227_VLLM_PIP_SPEC =', V227_VLLM_PIP_SPEC, flush=True)
        run_cmd([sys.executable, '-m', 'pip', 'install', '-q', V227_VLLM_PIP_SPEC], cwd='/content', log_path=OUT_ROOT / 'pip_install_vllm.log', check=True, heartbeat_s=60)
        if verify_import('vllm', 'verify_import_vllm.log') != 0:
            raise RuntimeError('vLLM import still failed after install.')
    print('=== V227 VLLM EVAL DEPENDENCY CHECK END ===', flush=True)

def is_complete_adapter_dir(path):
    path = pathlib.Path(path)
    return path.is_dir() and (path / 'adapter_config.json').exists() and (
        (path / 'adapter_model.safetensors').exists() or (path / 'adapter_model.bin').exists()
    )

print('=== V227 HELPERS END ===', flush=True)


=== V227 HELPERS START ===
=== V227 HELPERS END ===


In [4]:
# CELL: clone repo, compile scripts, and validate static data hashes.
print('=== V227 REPO SETUP START ===', flush=True)
if ROOT.exists():
    shutil.rmtree(ROOT)
run_cmd(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(ROOT)], cwd='/content', log_path=OUT_ROOT / 'repo_clone.log', check=True)
repo_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=str(ROOT), text=True).strip()
print('repo_commit =', repo_commit, flush=True)
compile_targets = [
    ROOT / 'src/competition_utils.py',
    ROOT / 'scripts/evaluate_lora_adapter.py',
    ROOT / 'scripts/evaluate_lora_adapters_batch.py',
    ROOT / 'scripts/hf_job_train_v90.py',
    ROOT / 'scripts/analyze_v227_checkpoint_sweep.py',
    ROOT / 'scripts/notebook_release_gate.py',
]
for py_path in compile_targets:
    print('compile_target =', py_path, 'exists =', py_path.exists(), flush=True)
    if not py_path.exists():
        raise FileNotFoundError(py_path)
    import py_compile
    py_compile.compile(str(py_path), doraise=True)
    print('py_compile ok =', py_path.relative_to(ROOT), flush=True)
train_path = ROOT / 'data/v217/v217_short_answer_train.jsonl'
val_path = ROOT / 'data/v217/v217_short_answer_val.jsonl'
print('train_path =', train_path, 'exists =', train_path.exists(), flush=True)
print('val_path =', val_path, 'exists =', val_path.exists(), flush=True)
observed_train_sha256 = sha256_file(train_path)
observed_val_sha256 = sha256_file(val_path)
train_rows = sum(1 for _ in train_path.open('r', encoding='utf-8'))
val_rows = sum(1 for _ in val_path.open('r', encoding='utf-8'))
print('observed_train_sha256 =', observed_train_sha256, flush=True)
print('observed_val_sha256 =', observed_val_sha256, flush=True)
print('train_rows =', train_rows, flush=True)
print('val_rows =', val_rows, flush=True)
if observed_train_sha256 != EXPECTED_TRAIN_SHA256:
    raise RuntimeError(f'train sha mismatch: {observed_train_sha256} != {EXPECTED_TRAIN_SHA256}')
if observed_val_sha256 != EXPECTED_VAL_SHA256:
    raise RuntimeError(f'val sha mismatch: {observed_val_sha256} != {EXPECTED_VAL_SHA256}')
if train_rows < MIN_TRAIN_EXAMPLES:
    raise RuntimeError(f'train row count too low: {train_rows} < {MIN_TRAIN_EXAMPLES}')
if val_rows < MIN_VAL_EXAMPLES:
    raise RuntimeError(f'val row count too low: {val_rows} < {MIN_VAL_EXAMPLES}')
print('=== V227 REPO SETUP END ===', flush=True)


=== V227 REPO SETUP START ===
--- COMMAND START ---
cwd = /content
+ git clone --depth 1 --branch v227-targeted-equation-micro-sweep https://github.com/FELIPEACASTRO/KG1-NVIDIA.git /content/kg1
log_path = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/repo_clone.log
Cloning into '/content/kg1'...
returncode = 0
elapsed_s = 1.8
--- COMMAND END ---
repo_commit = dbe82b7acc0acdc6187f6c13c4779d84a4768704
compile_target = /content/kg1/src/competition_utils.py exists = True
py_compile ok = src/competition_utils.py
compile_target = /content/kg1/scripts/evaluate_lora_adapter.py exists = True
py_compile ok = scripts/evaluate_lora_adapter.py
compile_target = /content/kg1/scripts/evaluate_lora_adapters_batch.py exists = True
py_compile ok = scripts/evaluate_lora_adapters_batch.py
compile_target = /content/kg1/scripts/hf_job_train_v90.py exists = True
py_compile ok = scripts/hf_job_train_v90.py
compile_target = /content/kg1/scripts/analyze_v227_checkpoint_sweep.py

In [6]:
# CELL: runtime, Drive artifact, adapter, and dependency audit.
print('=== V227 RUNTIME ARTIFACT AUDIT START ===', flush=True)

torch_probe_path = OUT_ROOT / 'verify_torch_cuda.jsonl'
torch_code = "import json, torch; props=torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None; print(json.dumps({'torch': getattr(torch, '__version__', 'unknown'), 'cuda_available': torch.cuda.is_available(), 'gpu_name': props.name if props else '', 'gpu_total_gib': props.total_memory/1024**3 if props else 0.0}))"
run_cmd([sys.executable, '-c', torch_code], cwd='/content', log_path=torch_probe_path, check=True)

torch_probe = json.loads([line for line in torch_probe_path.read_text(encoding='utf-8').splitlines() if line.strip()][-1])
cuda_available = bool(torch_probe.get('cuda_available'))
gpu_name = str(torch_probe.get('gpu_name', ''))
gpu_total_gib = float(torch_probe.get('gpu_total_gib', 0.0))
content_free_gib = shutil.disk_usage('/content').free / 1024**3

print('cuda_available =', cuda_available, flush=True)
print('gpu_name =', gpu_name, flush=True)
print('gpu_total_gib =', round(gpu_total_gib, 2), flush=True)
print('content_free_gib =', round(content_free_gib, 2), flush=True)

if not cuda_available:
    raise RuntimeError('CUDA is required for V227 targeted equation micro-sweep.')
if gpu_total_gib < 70:
    raise RuntimeError(f'GPU memory too small for V227 sweep: {gpu_total_gib:.2f} GiB')
if content_free_gib < 40:
    raise RuntimeError(f'/content free disk too low: {content_free_gib:.2f} GiB')

for module_name in ['causal_conv1d', 'mamba_ssm']:
    try:
        module = importlib.import_module(module_name)
        print(module_name, 'version =', getattr(module, '__version__', 'unknown'), flush=True)
    except Exception as exc:
        print(module_name, 'import_warning =', repr(exc), flush=True)

print('V221_WEAK_CSV exists =', V221_WEAK_CSV.exists(), flush=True)
print('V221_BATCH_SUMMARY_JSON exists =', V221_BATCH_SUMMARY_JSON.exists(), flush=True)
print('V225_FINAL_MANIFEST exists =', V225_FINAL_MANIFEST.exists(), flush=True)
print('V223_TRAIN_OUT exists =', V223_TRAIN_OUT.exists(), flush=True)
print('V226_TRAIN_OUT exists =', V226_TRAIN_OUT.exists(), flush=True)
print('V226_BEST_CHECKPOINT =', V226_BEST_CHECKPOINT, 'complete =', is_complete_adapter_dir(V226_BEST_CHECKPOINT), flush=True)

for required_path in [V221_WEAK_CSV, V221_BATCH_SUMMARY_JSON]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

try:
    from safetensors import safe_open
except Exception:
    run_cmd(
        [sys.executable, '-m', 'pip', 'install', '-q', 'safetensors'],
        cwd='/content',
        log_path=OUT_ROOT / 'pip_install_safetensors.log',
        check=True,
    )
    from safetensors import safe_open

if not is_complete_adapter_dir(INIT_ADAPTER_DIR):
    print('configured INIT_ADAPTER_DIR incomplete; falling back to V194_ADAPTER:', INIT_ADAPTER_DIR, flush=True)
    INIT_ADAPTER_DIR = V194_ADAPTER

print('resolved_INIT_ADAPTER_DIR =', INIT_ADAPTER_DIR, 'complete =', is_complete_adapter_dir(INIT_ADAPTER_DIR), flush=True)

for label, adapter_path in [('V194', V194_ADAPTER), ('V217', V217_ADAPTER), ('INIT', INIT_ADAPTER_DIR)]:
    print(label, 'adapter path =', adapter_path, 'complete =', is_complete_adapter_dir(adapter_path), flush=True)

    if not is_complete_adapter_dir(adapter_path):
        raise RuntimeError(f'{label} adapter incomplete: {adapter_path}')

    cfg = read_json(adapter_path / 'adapter_config.json')
    target_modules = cfg.get('target_modules') or []
    target_parameters = cfg.get('target_parameters') or []

    print(label, 'target_modules =', cfg.get('target_modules'), flush=True)
    print(label, 'target_parameters =', cfg.get('target_parameters'), flush=True)

    if sorted(target_modules) != sorted(EXPECTED_V194_TARGET_MODULES):
        raise RuntimeError(f'{label} target_modules mismatch')

    if label in {'V194', 'V217'}:
        if sorted(target_parameters) != sorted(EXPECTED_V194_TARGET_PARAMETERS):
            raise RuntimeError(f'{label} target_parameters mismatch')
    else:
        if sorted(target_parameters) != sorted(EXPECTED_V194_TARGET_PARAMETERS):
            print(
                label,
                'target_parameters differ from V194/V217; accepting PEFT checkpoint INIT format.',
                flush=True,
            )

    weights_path = adapter_path / 'adapter_model.safetensors'
    with safe_open(str(weights_path), framework='pt', device='cpu') as handle:
        tensor_count = len(handle.keys())

    weight_bytes = weights_path.stat().st_size
    print(label, 'adapter_tensor_count =', tensor_count, flush=True)
    print(label, 'adapter_weight_bytes =', weight_bytes, flush=True)

    if label == 'V194':
        if tensor_count != EXPECTED_V194_ADAPTER_TENSOR_COUNT:
            raise RuntimeError('V194 adapter tensor count mismatch')
        if weight_bytes != EXPECTED_V194_ADAPTER_BYTES:
            raise RuntimeError('V194 adapter weight size mismatch')

    if label == 'V217':
        if tensor_count < MIN_V217_ADAPTER_TENSOR_COUNT:
            raise RuntimeError('V217 final adapter tensor count below expected floor')
        if weight_bytes < MIN_V217_ADAPTER_BYTES:
            raise RuntimeError('V217 final adapter size mismatch')

print('=== V227 RUNTIME ARTIFACT AUDIT END ===', flush=True)


=== V227 RUNTIME ARTIFACT AUDIT START ===
--- COMMAND START ---
cwd = /content
+ /usr/bin/python3 -c import json, torch; props=torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None; print(json.dumps({'torch': getattr(torch, '__version__', 'unknown'), 'cuda_available': torch.cuda.is_available(), 'gpu_name': props.name if props else '', 'gpu_total_gib': props.total_memory/1024**3 if props else 0.0}))
log_path = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/verify_torch_cuda.jsonl
{"torch": "2.10.0+cu128", "cuda_available": true, "gpu_name": "NVIDIA H100 80GB HBM3", "gpu_total_gib": 79.1788330078125}
returncode = 0
elapsed_s = 1.5
--- COMMAND END ---
cuda_available = True
gpu_name = NVIDIA H100 80GB HBM3
gpu_total_gib = 79.18
content_free_gib = 192.74
causal_conv1d import_warning = ModuleNotFoundError("No module named 'causal_conv1d'")
mamba_ssm import_warning = ModuleNotFoundError("No module named 'mamba_ssm'")
V221_WEAK_CSV exists 

In [7]:
# CELL: prepare weak eval CSV and baseline checkpoint metadata.
print('=== V227 WEAK DATA PREP START ===', flush=True)
import pandas as pd
weak_df = pd.read_csv(V221_WEAK_CSV)
if 'id' not in weak_df.columns:
    raise RuntimeError('V221 weak CSV must include id column.')
if 'prompt' not in weak_df.columns:
    raise RuntimeError('V221 weak CSV must include prompt column.')
family_col = 'type' if 'type' in weak_df.columns else 'family'
if family_col not in weak_df.columns:
    raise RuntimeError('V221 weak CSV must include type/family column.')
equation_df = weak_df[weak_df[family_col].astype(str).eq('equation_transform')].copy()
bit_df = weak_df[weak_df[family_col].astype(str).eq('bit_manipulation')].copy()
print('weak_rows =', len(weak_df), flush=True)
print('equation_rows =', len(equation_df), flush=True)
print('bit_rows =', len(bit_df), flush=True)
if len(weak_df) != 315:
    raise RuntimeError(f'unexpected weak row count: {len(weak_df)} != 315')
if len(equation_df) != 155:
    raise RuntimeError(f'unexpected equation row count: {len(equation_df)} != 155')
if len(bit_df) != 160:
    raise RuntimeError(f'unexpected bit row count: {len(bit_df)} != 160')
WEAK_EVAL_CSV = EVAL_OUT / 'v227_weak_315.csv'
weak_df.to_csv(WEAK_EVAL_CSV, index=False)
if V225_FINAL_MANIFEST.exists():
    v225_manifest = read_json(V225_FINAL_MANIFEST)
    print('v225_final_decision =', json.dumps(v225_manifest.get('decision', {}), sort_keys=True), flush=True)
else:
    print('v225_final_decision = missing manifest; continuing with V227 targeted equation micro-sweep.', flush=True)
print('weak_eval_csv =', WEAK_EVAL_CSV, flush=True)
print('=== V227 WEAK DATA PREP END ===', flush=True)


=== V227 WEAK DATA PREP START ===
weak_rows = 315
equation_rows = 155
bit_rows = 160
v225_final_decision = {"best_candidate": "v217_final_existing", "best_variant": "think_strict_boxed", "decision": "no_equation_decode_candidate_passed_weak_gate", "next_action": "Build V226 targeted equation data/training or inspect gained/lost rows from V225.", "reason": "best_eq_correct=56; eq_gap=4; total_gap=2; trunc_gap=0"}
weak_eval_csv = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/eval_v227_targeted_equation_micro_sweep/v227_weak_315.csv
=== V227 WEAK DATA PREP END ===


In [8]:
# CELL: enable V227 micro training.
print('=== V227 ENABLE TRAIN START ===', flush=True)
os.environ['KG1_V227_RUN_TRAIN'] = '1'
RUN_TRAIN = os.environ.get('KG1_V227_RUN_TRAIN', '0').strip().lower() in {'1', 'true', 'yes', 'on'}
print('KG1_V227_RUN_TRAIN =', os.environ.get('KG1_V227_RUN_TRAIN'), flush=True)
print('RUN_TRAIN =', RUN_TRAIN, flush=True)
if not RUN_TRAIN:
    raise RuntimeError('RUN_TRAIN should be True after enabling KG1_V227_RUN_TRAIN=1.')
print('=== V227 ENABLE TRAIN END ===', flush=True)


=== V227 ENABLE TRAIN START ===
KG1_V227_RUN_TRAIN = 1
RUN_TRAIN = True
=== V227 ENABLE TRAIN END ===


In [9]:
# CELL: optional V227 targeted continuation from V226 checkpoint-1.
print('=== V227 TRAIN START ===', flush=True)
if not RUN_TRAIN:
    print('RUN_TRAIN is false; skipping V227 targeted micro training and evaluating existing checkpoints only.', flush=True)
else:
    final_adapter = TRAIN_OUT / 'final_adapter'
    run_new_train = FORCE_RETRAIN or not is_complete_adapter_dir(final_adapter)
    print('train_out =', TRAIN_OUT, flush=True)
    print('final_adapter =', final_adapter, 'complete =', is_complete_adapter_dir(final_adapter), flush=True)
    print('run_new_train =', run_new_train, flush=True)
    if run_new_train:
        ensure_train_dependencies()
        train_overrides = {
            'MODEL_NAME': MODEL_NAME,
            'MODEL_REVISION': 'cbd3fa9f933d55ef16a84236559f4ee2a0526848',
            'DATA_FILE': 'data/v217/v217_short_answer_train.jsonl',
            'VAL_FILE': 'data/v217/v217_short_answer_val.jsonl',
            'EXPECTED_TRAIN_SHA256': EXPECTED_TRAIN_SHA256,
            'EXPECTED_VAL_SHA256': EXPECTED_VAL_SHA256,
            'MIN_TRAIN_EXAMPLES': str(MIN_TRAIN_EXAMPLES),
            'MIN_VAL_EXAMPLES': str(MIN_VAL_EXAMPLES),
            'MIN_TOKENIZED_TRAIN_EXAMPLES': '10000',
            'MIN_TOKENIZED_VAL_EXAMPLES': '681',
            'OUTPUT_DIR': str(TRAIN_OUT),
            'OUTPUT_REPO': '',
            'UPLOAD_TO_HF': '0',
            'UPLOAD_CHECKPOINTS_DURING_TRAINING': '0',
            'INIT_ADAPTER_DIR': str(INIT_ADAPTER_DIR),
            'INIT_ADAPTER_LOAD_MODE': 'manual',
            'FAIL_ON_MISSING_ADAPTER_KEYS': '1',
            'TRAINABLE_LORA_MODULES': V227_TRAINABLE_MODULES,
            'MAX_TRAINABLE_PARAM_RATIO': '0.030',
            'MAX_LENGTH': str(V227_MAX_LENGTH),
            'BATCH_SIZE': str(V227_BATCH_SIZE),
            'MICRO_BATCH_SIZE': str(V227_MICRO_BATCH_SIZE),
            'LEARNING_RATE': V227_LR,
            'FINAL_LEARNING_RATE': V227_FINAL_LR,
            'MAX_STEPS': V227_MAX_STEPS,
            'SEED': V227_TRAIN_SEED,
            'NUM_EPOCHS': '1',
            'SAVE_EVERY_STEPS': '1',
            'EVAL_EVERY_STEPS': '1',
            'EVAL_MAX_EXAMPLES': '256',
            'LOG_EVERY_STEPS': '1',
            'MICRO_LOG_EVERY': '0',
            'BASELINE_EVAL_BEFORE_TRAIN': '1',
            'ABORT_EVAL_RELATIVE_TO_BASELINE_DELTA': '0.020',
            'MAX_FINAL_EVAL_REGRESSION': '0.020',
            'REQUIRE_FINAL_EVAL_LTE_BASELINE': '0',
            'ABORT_TRAIN_RISE_POINTS': '0',
            'SAMPLING_MODE': 'weighted_replacement',
            'SUBCATEGORY_WEIGHTS': 'equation_transform=2.05,equation_symbolic=2.00,equation_numeric=1.55,bit_manipulation=0.95',
            'SOURCE_WEIGHTS': 'v216=1.20,v217=1.10',
            'MAX_PROMPT_TRUNCATION_RATE': str(MAX_PROMPT_TRUNCATION_RATE),
            'REQUIRE_OFFSET_MASK': '1' if REQUIRE_OFFSET_MASK else '0',
            'TOKENIZE_ONLY_DRY_RUN': '0',
            'DRY_RUN_VALIDATE_ONLY': '0',
            'USE_BITSANDBYTES': '1',
            'RUN_ID': 'v227-from-v226ckpt1-eqnudge-lr5e10-s2',
        }
        for key, value in train_overrides.items():
            os.environ[key] = str(value)
        print('train_overrides =', json.dumps(train_overrides, indent=2, sort_keys=True), flush=True)
        rc = run_cmd([sys.executable, str(ROOT / 'scripts/hf_job_train_v90.py')], cwd=ROOT, log_path=TRAIN_OUT / 'v227_eqnudge_train.log', check=True, heartbeat_s=60, suppress_after_lines=320)
        print('v227 train returncode =', rc, flush=True)
    else:
        print('reusing existing V227 final adapter:', final_adapter, flush=True)
print('=== V227 TRAIN END ===', flush=True)


=== V227 TRAIN START ===
train_out = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/train_v227_from_v226ckpt1_eq_nudge_lr5e10_s2
final_adapter = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/train_v227_from_v226ckpt1_eq_nudge_lr5e10_s2/final_adapter complete = False
run_new_train = True
=== V227 TRAIN DEPENDENCY CHECK START ===
--- COMMAND START ---
cwd = /content
+ /usr/bin/python3 -c import importlib; m=importlib.import_module('causal_conv1d'); print('causal_conv1d' + ' subprocess_version=' + str(getattr(m, '__version__', 'unknown')))
log_path = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/verify_import_causal_conv1d.log
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [10]:
# CELL: collect V194, V217, V223, V226, and V227 checkpoint candidates.
print('=== V227 CANDIDATE COLLECTION START ===', flush=True)
candidates = []
seen = set()

def add_adapter(name, adapter_path, source_kind):
    adapter_path = pathlib.Path(adapter_path)
    complete = is_complete_adapter_dir(adapter_path)
    print('candidate_probe =', json.dumps({'name': name, 'adapter': str(adapter_path), 'source_kind': source_kind, 'complete': complete}, sort_keys=True), flush=True)
    if not complete:
        return
    key = str(adapter_path.resolve())
    if key in seen:
        return
    seen.add(key)
    candidates.append({'name': name, 'adapter': str(adapter_path), 'source_kind': source_kind})

add_adapter('v194_protected_baseline', V194_ADAPTER, 'baseline')
add_adapter('v217_final_existing', V217_ADAPTER, 'baseline')
if V223_TRAIN_OUT.exists():
    add_adapter('v223_final_adapter', V223_TRAIN_OUT / 'final_adapter', 'v223_final')
    for path in sorted(V223_TRAIN_OUT.glob('checkpoint-*')):
        add_adapter('v223_' + path.name.replace('-', '_'), path, 'v223_checkpoint')
else:
    print('V223_TRAIN_OUT missing; no V223 checkpoints collected.', flush=True)
if V226_TRAIN_OUT.exists():
    add_adapter('v226_best_checkpoint1_observed_191', V226_BEST_CHECKPOINT, 'v226_best')
    for path in sorted(V226_TRAIN_OUT.glob('checkpoint-*')):
        add_adapter('v226_' + path.name.replace('-', '_'), path, 'v226_checkpoint')
else:
    print('V226_TRAIN_OUT missing; no V226 checkpoints collected.', flush=True)
if TRAIN_OUT.exists():
    add_adapter('v227_final_adapter', TRAIN_OUT / 'final_adapter', 'v227_final')
    for path in sorted(TRAIN_OUT.glob('checkpoint-*')):
        add_adapter('v227_' + path.name.replace('-', '_'), path, 'v227_checkpoint')
if len(candidates) > V227_MAX_CANDIDATES:
    print('candidate_count_before_cap =', len(candidates), flush=True)
    baseline = [item for item in candidates if item['source_kind'] == 'baseline']
    rest = [item for item in candidates if item['source_kind'] != 'baseline']
    candidates = (baseline + rest)[:V227_MAX_CANDIDATES]
candidate_json = EVAL_OUT / 'v227_targeted_micro_candidates.json'
write_json(candidate_json, candidates)
print('candidate_count =', len(candidates), flush=True)
print('candidate_json =', candidate_json, flush=True)
print('candidate_rows =', json.dumps(candidates, indent=2, sort_keys=True), flush=True)
if len(candidates) < 2:
    raise RuntimeError('Need at least V194 and V217 candidates for V227 targeted equation micro-sweep.')
print('=== V227 CANDIDATE COLLECTION END ===', flush=True)


=== V227 CANDIDATE COLLECTION START ===
candidate_probe = {"adapter": "/content/drive/MyDrive/KG1_NVIDIA_V202D/init_adapter_v194_rank19_build/adapter", "complete": true, "name": "v194_protected_baseline", "source_kind": "baseline"}
candidate_probe = {"adapter": "/content/drive/MyDrive/KG1_NVIDIA_V217/output_v217_short_answer_rescue/train_v217_shortans_lr1e8_s16/final_adapter", "complete": true, "name": "v217_final_existing", "source_kind": "baseline"}
candidate_probe = {"adapter": "/content/drive/MyDrive/KG1_NVIDIA_V223/output_v223_equation_rescue/train_v223_eqrescue_from_v217_lr1e8_s12/final_adapter", "complete": true, "name": "v223_final_adapter", "source_kind": "v223_final"}
candidate_probe = {"adapter": "/content/drive/MyDrive/KG1_NVIDIA_V226/output_v226_equation_checkpoint_sweep/train_v226_v194_micro_lr2e9_s6/checkpoint-1", "complete": true, "name": "v226_best_checkpoint1_observed_191", "source_kind": "v226_best"}
candidate_probe = {"adapter": "/content/drive/MyDrive/KG1_NVIDIA_V2

In [11]:
# CELL: weak checkpoint batch eval with the best V225 prompt contract.
print('=== V227 WEAK CHECKPOINT EVAL START ===', flush=True)
weak_gate_pass_for_full = False
batch_summary_json = EVAL_OUT / 'batch_candidate_summary.json'
if not RUN_EVAL:
    print('RUN_EVAL is false; skipping V227 weak checkpoint eval.', flush=True)
else:
    ensure_vllm_for_eval()
    run_new_eval = FORCE_REEVAL or not batch_summary_json.exists()
    print('run_new_eval =', run_new_eval, flush=True)
    if run_new_eval:
        cmd = [
            sys.executable,
            str(ROOT / 'scripts/evaluate_lora_adapters_batch.py'),
            '--solution-csv', str(WEAK_EVAL_CSV),
            '--questions-csv', str(WEAK_EVAL_CSV),
            '--candidates-json', str(candidate_json),
            '--base-model-path', MODEL_NAME,
            '--label-prefix', 'v227_weak_ckpt',
            '--seed', '42',
            '--limit', '0',
            '--output-dir', str(EVAL_OUT),
            '--max-tokens', str(V227_MAX_TOKENS),
            '--max-model-len', str(V227_MAX_MODEL_LEN),
            '--max-num-seqs', str(V227_MAX_NUM_SEQS),
            '--warmup-rows', str(V227_WARMUP_ROWS),
            '--prompt-suffix', V227_PROMPT_SUFFIX,
            '--continue-on-error',
        ]
        rc = run_cmd(cmd, cwd=ROOT, log_path=EVAL_OUT / 'weak_checkpoint_eval.log', check=True, heartbeat_s=60, suppress_after_lines=360)
        print('weak checkpoint eval returncode =', rc, flush=True)
    else:
        print('reusing batch summary:', batch_summary_json, flush=True)
print('batch_summary_json =', batch_summary_json, flush=True)
print('=== V227 WEAK CHECKPOINT EVAL END ===', flush=True)


=== V227 WEAK CHECKPOINT EVAL START ===
=== V227 VLLM EVAL DEPENDENCY CHECK START ===
--- COMMAND START ---
cwd = /content
+ /usr/bin/python3 -c import importlib; m=importlib.import_module('vllm'); print('vllm' + ' subprocess_version=' + str(getattr(m, '__version__', 'unknown')))
log_path = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/verify_import_vllm.log
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1324, in _find_and_load_unlocked
ModuleNotFoundError: No module named 'vllm'
returncode = 1
elapsed_s = 0.0
command_tail_on_failure = Traceback 

KeyboardInterrupt: 

In [12]:
# CELL: V227 fast weak checkpoint eval with lower generation cap.
print('=== V227 FAST WEAK CHECKPOINT EVAL START ===', flush=True)

FAST_EVAL_OUT = OUT_ROOT / 'eval_v227_targeted_equation_micro_sweep_fast_mtok2048'
FAST_EVAL_OUT.mkdir(parents=True, exist_ok=True)

FAST_WEAK_EVAL_CSV = FAST_EVAL_OUT / 'v227_weak_315.csv'
weak_df.to_csv(FAST_WEAK_EVAL_CSV, index=False)

fast_batch_summary_json = FAST_EVAL_OUT / 'batch_candidate_summary.json'

ensure_vllm_for_eval()

run_new_fast_eval = FORCE_REEVAL or not fast_batch_summary_json.exists()
print('FAST_EVAL_OUT =', FAST_EVAL_OUT, flush=True)
print('fast_batch_summary_json =', fast_batch_summary_json, flush=True)
print('run_new_fast_eval =', run_new_fast_eval, flush=True)
print('candidate_json =', candidate_json, flush=True)
print('FAST_WEAK_EVAL_CSV =', FAST_WEAK_EVAL_CSV, flush=True)

if run_new_fast_eval:
    cmd = [
        sys.executable,
        str(ROOT / 'scripts/evaluate_lora_adapters_batch.py'),
        '--solution-csv', str(FAST_WEAK_EVAL_CSV),
        '--questions-csv', str(FAST_WEAK_EVAL_CSV),
        '--candidates-json', str(candidate_json),
        '--base-model-path', MODEL_NAME,
        '--label-prefix', 'v227_fast_weak_ckpt',
        '--seed', '42',
        '--limit', '0',
        '--output-dir', str(FAST_EVAL_OUT),
        '--max-tokens', '2048',
        '--max-model-len', str(V227_MAX_MODEL_LEN),
        '--max-num-seqs', '32',
        '--warmup-rows', '0',
        '--prompt-suffix', V227_PROMPT_SUFFIX,
        '--continue-on-error',
    ]
    rc = run_cmd(
        cmd,
        cwd=ROOT,
        log_path=FAST_EVAL_OUT / 'fast_weak_checkpoint_eval.log',
        check=True,
        heartbeat_s=60,
        suppress_after_lines=360,
    )
    print('fast weak checkpoint eval returncode =', rc, flush=True)
else:
    print('reusing fast batch summary:', fast_batch_summary_json, flush=True)

print('fast_batch_summary_json =', fast_batch_summary_json, flush=True)
print('=== V227 FAST WEAK CHECKPOINT EVAL END ===', flush=True)


=== V227 FAST WEAK CHECKPOINT EVAL START ===
=== V227 VLLM EVAL DEPENDENCY CHECK START ===
--- COMMAND START ---
cwd = /content
+ /usr/bin/python3 -c import importlib; m=importlib.import_module('vllm'); print('vllm' + ' subprocess_version=' + str(getattr(m, '__version__', 'unknown')))
log_path = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/verify_import_vllm.log
vllm subprocess_version=0.20.1
returncode = 0
elapsed_s = 2.1
--- COMMAND END ---
=== V227 VLLM EVAL DEPENDENCY CHECK END ===
FAST_EVAL_OUT = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/eval_v227_targeted_equation_micro_sweep_fast_mtok2048
fast_batch_summary_json = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/eval_v227_targeted_equation_micro_sweep_fast_mtok2048/batch_candidate_summary.json
run_new_fast_eval = True
candidate_json = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/eval_v227

KeyboardInterrupt: 

In [16]:
# CELL: V227 focused eval without V194 baseline.
print('=== V227 FOCUSED NO-V194 WEAK EVAL START ===', flush=True)

NO_V194_EVAL_OUT = OUT_ROOT / 'eval_v227_focused_v226_v227_no_v194_mtok1024'
NO_V194_EVAL_OUT.mkdir(parents=True, exist_ok=True)

NO_V194_WEAK_EVAL_CSV = NO_V194_EVAL_OUT / 'v227_weak_315.csv'
weak_df.to_csv(NO_V194_WEAK_EVAL_CSV, index=False)

no_v194_candidates_json = NO_V194_EVAL_OUT / 'v227_focused_no_v194_candidates.json'
no_v194_batch_summary_json = NO_V194_EVAL_OUT / 'batch_candidate_summary.json'

all_candidates = read_json(candidate_json)
if isinstance(all_candidates, dict):
    all_candidates = all_candidates.get('candidates', all_candidates.get('rows', []))

selected = []
for cand in all_candidates:
    name = str(cand.get('name', ''))
    adapter = str(cand.get('adapter', ''))
    keep = (
        name == 'v226_best_checkpoint1_observed_191'
        or name.startswith('v227')
        or '/KG1_NVIDIA_V227/' in adapter
    )
    if keep and name != 'v194_protected_baseline':
        selected.append(cand)

seen = set()
deduped = []
for cand in selected:
    key = (cand.get('name'), cand.get('adapter'))
    if key not in seen:
        seen.add(key)
        deduped.append(cand)

if not deduped:
    raise RuntimeError('No V226/V227 candidates found. Check candidate_json contents.')

no_v194_candidates_json.write_text(json.dumps(deduped, indent=2, sort_keys=True), encoding='utf-8')

print('selected candidate count =', len(deduped), flush=True)
print('selected candidates =', json.dumps(deduped, indent=2, sort_keys=True), flush=True)
print('NO_V194_EVAL_OUT =', NO_V194_EVAL_OUT, flush=True)
print('no_v194_candidates_json =', no_v194_candidates_json, flush=True)

ensure_vllm_for_eval()

run_new_no_v194_eval = FORCE_REEVAL or not no_v194_batch_summary_json.exists()
print('run_new_no_v194_eval =', run_new_no_v194_eval, flush=True)

if run_new_no_v194_eval:
    cmd = [
        sys.executable,
        str(ROOT / 'scripts/evaluate_lora_adapters_batch.py'),
        '--solution-csv', str(NO_V194_WEAK_EVAL_CSV),
        '--questions-csv', str(NO_V194_WEAK_EVAL_CSV),
        '--candidates-json', str(no_v194_candidates_json),
        '--base-model-path', MODEL_NAME,
        '--label-prefix', 'v227_no_v194_weak',
        '--seed', '42',
        '--limit', '0',
        '--output-dir', str(NO_V194_EVAL_OUT),
        '--max-tokens', '1024',
        '--max-model-len', str(V227_MAX_MODEL_LEN),
        '--max-num-seqs', '16',
        '--warmup-rows', '0',
        '--prompt-suffix', V227_PROMPT_SUFFIX,
        '--continue-on-error',
    ]
    rc = run_cmd(
        cmd,
        cwd=ROOT,
        log_path=NO_V194_EVAL_OUT / 'no_v194_weak_eval.log',
        check=True,
        heartbeat_s=60,
        suppress_after_lines=360,
    )
    print('no-v194 weak eval returncode =', rc, flush=True)
else:
    print('reusing no-v194 batch summary:', no_v194_batch_summary_json, flush=True)

print('no_v194_batch_summary_json =', no_v194_batch_summary_json, flush=True)
print('=== V227 FOCUSED NO-V194 WEAK EVAL END ===', flush=True)


=== V227 FOCUSED NO-V194 WEAK EVAL START ===
selected candidate count = 3
selected candidates = [
  {
    "adapter": "/content/drive/MyDrive/KG1_NVIDIA_V226/output_v226_equation_checkpoint_sweep/train_v226_v194_micro_lr2e9_s6/checkpoint-1",
    "name": "v226_best_checkpoint1_observed_191",
    "source_kind": "v226_best"
  },
  {
    "adapter": "/content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/train_v227_from_v226ckpt1_eq_nudge_lr5e10_s2/final_adapter",
    "name": "v227_final_adapter",
    "source_kind": "v227_final"
  },
  {
    "adapter": "/content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/train_v227_from_v226ckpt1_eq_nudge_lr5e10_s2/checkpoint-1",
    "name": "v227_checkpoint_1",
    "source_kind": "v227_checkpoint"
  }
]
NO_V194_EVAL_OUT = /content/drive/MyDrive/KG1_NVIDIA_V227/output_v227_targeted_equation_micro_sweep/eval_v227_focused_v226_v227_no_v194_mtok1024
no_v194_candidates_json = /content/drive/MyDrive/KG1_NVIDI

KeyboardInterrupt: 

In [ ]:
# CELL: analyze V227 fast weak checkpoint eval.
print('=== V227 FAST CHECKPOINT ANALYSIS START ===', flush=True)

FAST_ANALYSIS_OUT = OUT_ROOT / 'analysis_v227_targeted_equation_micro_sweep_fast_mtok2048'
FAST_ANALYSIS_OUT.mkdir(parents=True, exist_ok=True)

fast_analysis_manifest_path = FAST_ANALYSIS_OUT / 'v227_fast_targeted_equation_micro_sweep_manifest.json'

cmd = [
    sys.executable,
    str(ROOT / 'scripts/analyze_v227_checkpoint_sweep.py'),
    '--batch-summary-json', str(fast_batch_summary_json),
    '--output-dir', str(FAST_ANALYSIS_OUT),
    '--label', 'v227_fast_targeted_equation_micro_sweep',
    '--weak-total-min', str(WEAK_MIN_FOR_FULL),
    '--weak-eq-min', str(WEAK_EQ_MIN_FOR_FULL),
    '--weak-bit-min', str(WEAK_BIT_MIN_FOR_FULL),
    '--weak-trunc-max', str(WEAK_MAX_TRUNC_FOR_FULL),
]
rc = run_cmd(cmd, cwd=ROOT, log_path=FAST_ANALYSIS_OUT / 'v227_fast_targeted_equation_micro_sweep.log', check=True)
print('fast analysis returncode =', rc, flush=True)

fast_analysis_manifest = read_json(fast_analysis_manifest_path)
print('decision =', json.dumps(fast_analysis_manifest.get('decision', {}), indent=2, sort_keys=True), flush=True)
print('best =', json.dumps(fast_analysis_manifest.get('best', {}), indent=2, sort_keys=True)[:6000], flush=True)
print('fast_analysis_manifest_path =', fast_analysis_manifest_path, flush=True)

print('=== V227 FAST CHECKPOINT ANALYSIS END ===', flush=True)


In [ ]:
# CELL: analyze V227 targeted micro sweep and decide gate action.
print('=== V227 TARGETED MICRO ANALYSIS START ===', flush=True)
analysis_manifest_path = ANALYSIS_OUT / 'v227_targeted_equation_micro_sweep_manifest.json'
if not RUN_ANALYSIS:
    print('RUN_ANALYSIS is false; skipping analysis.', flush=True)
else:
    if not batch_summary_json.exists():
        raise FileNotFoundError(batch_summary_json)
    cmd = [
        sys.executable,
        str(ROOT / 'scripts/analyze_v227_checkpoint_sweep.py'),
        '--batch-summary-json', str(batch_summary_json),
        '--output-dir', str(ANALYSIS_OUT),
        '--label', 'v227_targeted_equation_micro_sweep',
        '--weak-total-min', str(WEAK_MIN_FOR_FULL),
        '--weak-eq-min', str(WEAK_EQ_MIN_FOR_FULL),
        '--weak-bit-min', str(WEAK_BIT_MIN_FOR_FULL),
        '--weak-trunc-max', str(WEAK_MAX_TRUNC_FOR_FULL),
    ]
    rc = run_cmd(cmd, cwd=ROOT, log_path=ANALYSIS_OUT / 'v227_targeted_equation_micro_sweep.log', check=True)
    print('v227 analysis returncode =', rc, flush=True)
analysis_manifest = read_json(analysis_manifest_path)
decision = analysis_manifest.get('decision', {})
best = analysis_manifest.get('best', {})
weak_gate_pass_for_full = bool(best.get('weak_gate_pass_for_full', False))
print('analysis_manifest_path =', analysis_manifest_path, flush=True)
print('decision =', json.dumps(decision, indent=2, sort_keys=True), flush=True)
print('best =', json.dumps(best, indent=2, sort_keys=True)[:6000], flush=True)
print('weak_gate_pass_for_full =', weak_gate_pass_for_full, flush=True)
print('=== V227 TARGETED MICRO ANALYSIS END ===', flush=True)


In [ ]:
# CELL: full eval/package hard block and final manifest.
print('=== V227 FINAL MANIFEST START ===', flush=True)
analysis_manifest = read_json(ANALYSIS_OUT / 'v227_targeted_equation_micro_sweep_manifest.json')
decision = analysis_manifest.get('decision', {})
best = analysis_manifest.get('best', {})
weak_gate_pass_for_full = bool(best.get('weak_gate_pass_for_full', False))
full_candidate_gate = False
print('weak_gate_pass_for_full =', weak_gate_pass_for_full, flush=True)
print('full_candidate_gate =', full_candidate_gate, flush=True)
print('Required weak_total >=', WEAK_MIN_FOR_FULL, 'eq >=', WEAK_EQ_MIN_FOR_FULL, 'bit >=', WEAK_BIT_MIN_FOR_FULL, 'trunc <=', WEAK_MAX_TRUNC_FOR_FULL, flush=True)
print('Full eval is blocked by default to avoid accidental GPU spend.', flush=True)
print('Full eval is intentionally not automatic in V227 targeted equation micro-sweep notebook.', flush=True)
print('No package and no Kaggle submit can be created in V227.', flush=True)
if RUN_FULL_IF_GATE or ALLOW_KAGGLE_SUBMIT:
    raise RuntimeError('V227 hard block violated. Kaggle submission is disabled.')
final_manifest_path = OUT_ROOT / 'v227_targeted_equation_micro_sweep_final_manifest.json'
final_manifest = {
    'version': VERSION,
    'repo_branch': REPO_BRANCH,
    'weak_gate_pass_for_full': weak_gate_pass_for_full,
    'full_candidate_gate': full_candidate_gate,
    'decision': decision,
    'best': best,
    'thresholds': {
        'weak_total': WEAK_MIN_FOR_FULL,
        'weak_equation_transform': WEAK_EQ_MIN_FOR_FULL,
        'weak_bit_manipulation': WEAK_BIT_MIN_FOR_FULL,
        'weak_truncated': WEAK_MAX_TRUNC_FOR_FULL,
        'full_min_candidate': FULL_MIN_CANDIDATE,
        'full_max_trunc': FULL_MAX_TRUNC,
    },
    'train_out': str(TRAIN_OUT),
    'eval_out': str(EVAL_OUT),
    'analysis_manifest': str(ANALYSIS_OUT / 'v227_targeted_equation_micro_sweep_manifest.json'),
    'roadmap_next': decision.get('next_action', 'Review V227 targeted equation micro-sweep outputs.'),
}
write_json(final_manifest_path, final_manifest)
print('final_manifest_path =', final_manifest_path, flush=True)
print('final_decision =', json.dumps(decision, indent=2, sort_keys=True), flush=True)
print('roadmap_next =', final_manifest['roadmap_next'], flush=True)
print('=== V227 FINAL MANIFEST END ===', flush=True)
